In [2]:
# Environment Setup and Imports


# Standard library
from pathlib import Path
import random
import time
import sys


# Numerical computing
import numpy as np
import pandas as pd


# Visualization
import matplotlib.pyplot as plt


# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim


# Computer vision
import torchvision
from torchvision import datasets, transforms, models


# Data loading
from torch.utils.data import DataLoader


# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


print("Environment initialized")

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"Torchvision : {torchvision.__version__}")


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print(f"Device      : {device}")

if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

Environment initialized
Python      : 3.12.3
PyTorch     : 2.12.1+cu130
Torchvision : 0.27.1+cu130
Device      : cuda
GPU         : NVIDIA GeForce RTX 5070 Ti


In [3]:
# Cell 2 — Reproducibility


SEED = 42


random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print(f"Random seed fixed: {SEED}")

Random seed fixed: 42


In [8]:
# Cell 3 — Dataset Paths


datasetRoot = Path("../dataset")


trainPath = datasetRoot / "train"
valPath   = datasetRoot / "val"
testPath  = datasetRoot / "test"


print("Dataset exists:", datasetRoot.exists())

print("Train:", trainPath.exists())
print("Val  :", valPath.exists())
print("Test :", testPath.exists())

Dataset exists: True
Train: True
Val  : True
Test : True


In [7]:
# Cell 4 — Image Transformations


# Training transformations with augmentation

trainTransforms = transforms.Compose(

    [

        transforms.Resize((224, 224)),


        transforms.RandomHorizontalFlip(),


        transforms.RandomRotation(10),


        transforms.ToTensor(),


        transforms.Normalize(

            mean=[0.485, 0.456, 0.406],

            std=[0.229, 0.224, 0.225]

        )

    ]

)



# Validation and testing transformations

evalTransforms = transforms.Compose(

    [

        transforms.Resize((224, 224)),


        transforms.ToTensor(),


        transforms.Normalize(

            mean=[0.485, 0.456, 0.406],

            std=[0.229, 0.224, 0.225]

        )

    ]

)



print("Transforms ready")

Transforms ready


In [9]:
# Cell 5 — Create PyTorch Datasets


# Load image datasets


trainDataset = datasets.ImageFolder(

    trainPath,

    transform=trainTransforms

)



valDataset = datasets.ImageFolder(

    valPath,

    transform=evalTransforms

)



testDataset = datasets.ImageFolder(

    testPath,

    transform=evalTransforms

)



# Dataset information


classNames = trainDataset.classes

numClasses = len(classNames)



print(
    f"Classes ({numClasses}):"
)


print(classNames)



print("\nDataset sizes:")


print(
    "Train:",
    len(trainDataset)
)


print(
    "Val:",
    len(valDataset)
)


print(
    "Test:",
    len(testDataset)
)

Classes (8):
['AMD', 'CNV', 'CSR', 'DME', 'DR', 'DRUSEN', 'MH', 'NORMAL']

Dataset sizes:
Train: 18400
Val: 2800
Test: 2800


In [10]:
# Cell 6 — DataLoaders


batchSize = 128



trainLoader = DataLoader(

    trainDataset,

    batch_size=batchSize,

    shuffle=True,

    num_workers=0

)



valLoader = DataLoader(

    valDataset,

    batch_size=batchSize,

    shuffle=False,

    num_workers=0

)



testLoader = DataLoader(

    testDataset,

    batch_size=batchSize,

    shuffle=False,

    num_workers=0

)



print("DataLoaders ready")



# Verify one batch


images, labels = next(
    iter(trainLoader)
)



print(
    "Batch shape:",
    images.shape
)

DataLoaders ready
Batch shape: torch.Size([128, 3, 224, 224])


In [ ]:
# Cell 7 — Load InceptionV3 Model


model = models.inception_v3(

    weights=models.Inception_V3_Weights.IMAGENET1K_V1

)


# Replace classifier

inputFeatures = model.fc.in_features


model.fc = nn.Linear(

    inputFeatures,

    numClasses

)


model = model.to(device)


print("InceptionV3 initialized")
print(model.fc)

ResNet50 initialized
Linear(in_features=2048, out_features=8, bias=True)


In [14]:
# Cell 8 — Loss Function and Optimizer


criterion = nn.CrossEntropyLoss()



optimizer = optim.Adam(

    model.parameters(),

    lr=1e-4

)



print("Training setup complete")

Training setup complete


In [16]:
# Cell 9 — Forward Pass Check


# Get one batch of images

images, labels = next(
    iter(trainLoader)
)



# Move batch to GPU

images = images.to(device)

labels = labels.to(device)



# Disable gradients for testing

model.eval()


with torch.no_grad():

    outputs = model(images)



# Display tensor dimensions


print(
    "Input shape :",
    images.shape
)


print(
    "Output shape:",
    outputs.shape
)


print(
    "Device:",
    images.device
)

Input shape : torch.Size([128, 3, 224, 224])
Output shape: torch.Size([128, 8])
Device: cuda:0


In [18]:
# Cell 10 — Training Function


def trainOneEpoch(
    model,
    dataLoader,
    criterion,
    optimizer,
    device
):


    # Enable training mode

    model.train()



    runningLoss = 0.0

    correctPredictions = 0

    totalSamples = 0



    # Iterate through batches

    for images, labels in dataLoader:


        # Move data to device

        images = images.to(device)

        labels = labels.to(device)



        # Reset gradients

        optimizer.zero_grad()



        # Forward pass

        outputs = model(images)



        # Compute loss

        loss = criterion(
            outputs,
            labels
        )



        # Backward pass

        loss.backward()



        # Update weights

        optimizer.step()



        # Track loss

        runningLoss += (
            loss.item()
            *
            images.size(0)
        )



        # Get predicted class

        _, predictions = torch.max(
            outputs,
            1
        )



        # Track accuracy

        correctPredictions += (
            predictions == labels
        ).sum().item()



        totalSamples += labels.size(0)



    # Calculate epoch metrics

    epochLoss = (
        runningLoss
        /
        totalSamples
    )


    epochAccuracy = (
        correctPredictions
        /
        totalSamples
    )



    return epochLoss, epochAccuracy

In [43]:
# Training function


def trainOneEpoch(model, dataLoader, criterion, optimizer, device):


    # Put model in training mode

    model.train()


    runningLoss = 0.0

    correctPredictions = 0

    totalSamples = 0



    for images, labels in dataLoader:


        # Move data to GPU/CPU

        images = images.to(device)

        labels = labels.to(device)



        # Clear previous gradients

        optimizer.zero_grad()



        # Forward propagation

        outputs = model(images)



        # Calculate loss

        loss = criterion(
            outputs,
            labels
        )



        # Backpropagation

        loss.backward()



        # Update model parameters

        optimizer.step()



        # Accumulate loss

        runningLoss += (
            loss.item() * images.size(0)
        )



        # Convert class scores to predictions

        _, predictions = torch.max(
            outputs,
            1
        )



        # Count correct predictions

        correctPredictions += (
            predictions == labels
        ).sum().item()



        totalSamples += labels.size(0)




    epochLoss = (
        runningLoss / totalSamples
    )


    epochAccuracy = (
        correctPredictions / totalSamples
    )



    return epochLoss, epochAccuracy

In [20]:
# Cell 11 — Validation Function


def validateModel(
    model,
    dataLoader,
    criterion,
    device
):


    # Enable evaluation mode

    model.eval()



    runningLoss = 0.0

    correctPredictions = 0

    totalSamples = 0



    # Disable gradient calculation

    with torch.no_grad():



        # Iterate through batches

        for images, labels in dataLoader:



            # Move data to device

            images = images.to(device)

            labels = labels.to(device)



            # Forward pass

            outputs = model(images)



            # Compute loss

            loss = criterion(

                outputs,

                labels

            )



            # Track loss

            runningLoss += (

                loss.item()

                *

                images.size(0)

            )



            # Get predicted class

            _, predictions = torch.max(

                outputs,

                1

            )



            # Track accuracy

            correctPredictions += (

                predictions == labels

            ).sum().item()



            totalSamples += labels.size(0)




    # Calculate validation metrics

    epochLoss = (

        runningLoss

        /

        totalSamples

    )



    epochAccuracy = (

        correctPredictions

        /

        totalSamples

    )



    return epochLoss, epochAccuracy

In [ ]:
# Cell 12 — Model Training


# Training configuration


numEpochs = 1



history = {

    "trainLoss": [],

    "trainAccuracy": [],

    "valLoss": [],

    "valAccuracy": []

}



bestValAccuracy = 0.0



# Create directory for model checkpoints


Path("../models").mkdir(

    exist_ok=True

)



# Start training


for epoch in range(numEpochs):



    startTime = time.time()



    # Training phase


    trainLoss, trainAccuracy = trainOneEpoch(

        model,

        trainLoader,

        criterion,

        optimizer,

        device

    )



    # Validation phase


    valLoss, valAccuracy = validateModel(

        model,

        valLoader,

        criterion,

        device

    )



    # Store training history


    history["trainLoss"].append(

        trainLoss

    )


    history["trainAccuracy"].append(

        trainAccuracy

    )


    history["valLoss"].append(

        valLoss

    )


    history["valAccuracy"].append(

        valAccuracy

    )




    # Save best model


    if valAccuracy > bestValAccuracy:



        bestValAccuracy = valAccuracy



        torch.save(

            model.state_dict(),

          "../models/bestInceptionV3.pth"

        )




    epochTime = (

        time.time()

        -

        startTime

    )



    # Display epoch results


    print(

        f"Epoch [{epoch + 1}/{numEpochs}] "

        f"| Train Loss: {trainLoss:.4f} "

        f"| Train Acc: {trainAccuracy:.4f} "

        f"| Val Loss: {valLoss:.4f} "

        f"| Val Acc: {valAccuracy:.4f} "

        f"| Time: {epochTime:.2f}s"

    )




print("\nTraining complete")


print(

    f"Best validation accuracy: {bestValAccuracy:.4f}"

)

Epoch [1/1] | Train Loss: 0.4475 | Train Acc: 0.8652 | Val Loss: 0.1127 | Val Acc: 0.9618 | Time: 415.65s

Training complete
Best validation accuracy: 0.9618


In [ ]:
# Cell 12 — Test Evaluation


# Load best saved model

model.load_state_dict(
    torch.load(
        "../models/bestInceptionV3.pth"
    )
)


model = model.to(device)


model.eval()



allPredictions = []

allLabels = []



with torch.no_grad():


    for images, labels in testLoader:


        images = images.to(device)

        labels = labels.to(device)



        outputs = model(images)



        _, predictions = torch.max(
            outputs,
            1
        )



        allPredictions.extend(
            predictions.cpu().numpy()
        )


        allLabels.extend(
            labels.cpu().numpy()
        )




# Metrics


testAccuracy = accuracy_score(
    allLabels,
    allPredictions
)


testPrecision = precision_score(
    allLabels,
    allPredictions,
    average="weighted"
)


testRecall = recall_score(
    allLabels,
    allPredictions,
    average="weighted"
)


testF1 = f1_score(
    allLabels,
    allPredictions,
    average="weighted"
)




print("Test Results")
print("---------------------")

print(
    f"Accuracy : {testAccuracy:.4f}"
)

print(
    f"Precision: {testPrecision:.4f}"
)

print(
    f"Recall   : {testRecall:.4f}"
)

print(
    f"F1 Score : {testF1:.4f}"
)

Test Results
---------------------
Accuracy : 0.9611
Precision: 0.9617
Recall   : 0.9611
F1 Score : 0.9610
